<a href="https://colab.research.google.com/github/abansal31868/Miscellaneous-/blob/main/Autoencoder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models

# ---------------------------------------------------------
# 1. Load and Preprocess Fashion-MNIST Data
# ---------------------------------------------------------
(x_train, _), (x_test, _) = tf.keras.datasets.fashion_mnist.load_data()

# Normalize pixel values from [0, 255] to [0.0, 1.0]
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

# Add channel dimension: (batch, 28, 28) -> (batch, 28, 28, 1)
x_train = np.expand_dims(x_train, axis=-1)
x_test = np.expand_dims(x_test, axis=-1)

# ---------------------------------------------------------
# 2. Build the Encoder and Decoder
# ---------------------------------------------------------
latent_dim = 64

# --- Encoder ---
encoder_inputs = layers.Input(shape=(28, 28, 1), name="encoder_input")
x = layers.Conv2D(32, (3, 3), activation="relu", strides=2, padding="same")(
    encoder_inputs
)  # 14x14x32
x = layers.Conv2D(64, (3, 3), activation="relu", strides=2, padding="same")(
    x
)  # 7x7x64
x = layers.Flatten()(x)
latent_vector = layers.Dense(latent_dim, activation="relu", name="latent_space")(
    x
)

encoder = models.Model(encoder_inputs, latent_vector, name="encoder")
encoder.summary()

# --- Decoder ---
decoder_inputs = layers.Input(shape=(latent_dim,), name="decoder_input")
x = layers.Dense(7 * 7 * 64, activation="relu")(decoder_inputs)
x = layers.Reshape((7, 7, 64))(x)
x = layers.Conv2DTranspose(
    64, (3, 3), activation="relu", strides=2, padding="same"
)(
    x
)  # 14x14x64
x = layers.Conv2DTranspose(
    32, (3, 3), activation="relu", strides=2, padding="same"
)(
    x
)  # 28x28x32
decoder_outputs = layers.Conv2D(
    1, (3, 3), activation="sigmoid", padding="same", name="reconstruction"
)(
    x
)  # 28x28x1

decoder = models.Model(decoder_inputs, decoder_outputs, name="decoder")
decoder.summary()

# --- Combined Autoencoder ---
autoencoder_inputs = encoder_inputs
encoded = encoder(autoencoder_inputs)
decoded = decoder(encoded)

autoencoder = models.Model(
    autoencoder_inputs, decoded, name="conv_autoencoder"
)
autoencoder.compile(optimizer="adam", loss="binary_crossentropy")

# ---------------------------------------------------------
# 3. Train the Model
# ---------------------------------------------------------
history = autoencoder.fit(
    x_train,
    x_train,
    epochs=15,
    batch_size=128,
    shuffle=True,
    validation_data=(x_test, x_test),
)

# ---------------------------------------------------------
# 4. Evaluate and Reconstruct Images
# ---------------------------------------------------------
decoded_imgs = autoencoder.predict(x_test)

# Plot original vs. reconstructed images side by side
n = 10  # Number of samples to visualize
plt.figure(figsize=(20, 4))

for i in range(n):
    # Original
    ax = plt.subplot(2, n, i + 1)
    plt.imshow(x_test[i].reshape(28, 28), cmap="gray")
    plt.title("Original")
    plt.axis("off")

    # Reconstruction
    ax = plt.subplot(2, n, i + 1 + n)
    plt.imshow(decoded_imgs[i].reshape(28, 28), cmap="gray")
    plt.title("Reconstructed")
    plt.axis("off")

plt.tight_layout()
plt.show()

29515/29515 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
26421880/26421880 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
5148/5148 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
4422102/4422102 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "encoder"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ encoder_input (InputLayer)      │ (None, 28, 28, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 14, 14, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 7, 7, 64)       │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 3136)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ latent_space (Dense)            │ (None, 64)             │       200,768 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 219,584 (857.75 KB)

 Trainable params: 219,584 (857.75 KB)

...

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ decoder_input (InputLayer)      │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 3136)           │       203,840 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reshape (Reshape)               │ (None, 7, 7, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_transpose                │ (None, 14, 14, 64)     │        36,928 │
│ (Conv2DTranspose)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_transpose_1              │ (None, 28, 28, 32)     │        18,464 │
│ (Conv2DTranspose)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reconstruction (Conv2D)         │ (None, 28, 28, 1)      │           289 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 259,521 (1013.75 KB)

 Trainable params: 259,521 (1013.75 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/15
469/469 ━━━━━━━━━━━━━━━━━━━━ 79s 164ms/step - loss: 0.3326 - val_loss: 0.2882
Epoch 2/15
469/469 ━━━━━━━━━━━━━━━━━━━━ 78s 156ms/step - loss: 0.2802 - val_loss: 0.2785
Epoch 3/15
469/469 ━━━━━━━━━━━━━━━━━━━━ 75s 159ms/step - loss: 0.2734 - val_loss: 0.2732
Epoch 4/15
469/469 ━━━━━━━━━━━━━━━━━━━━ 75s 159ms/step - loss: 0.2698 - val_loss: 0.2707
Epoch 5/15
266/469 ━━━━━━━━━━━━━━━━━━━━ 31s 153ms/step - loss: 0.2682